# Scoring Check

This notebook validates phase 03 against small synthetic geometry cases first, then runs the real workspace scoring flow once `grid_cell_size_m` is populated in the traced floor-plan metadata.


In [ ]:
from pathlib import Path

import numpy as np
from matplotlib import pyplot as plt
from matplotlib.patches import Wedge

from src.common.floorplan import FloorPlanInput, NULL_CELL, OPEN_CELL, SOLID_CELL
from src.planner._shared.config import PlannerConfig
from src.planner.main import load_workspace
from src.planner.phase01_candidate_generation import (
    PHASE_ARTIFACT_STEM as PHASE01_ARTIFACT_STEM,
    generate_candidate_generation_artifacts,
    load_candidate_generation_artifacts,
    save_candidate_generation_artifacts,
    validate_candidate_generation_artifacts,
)
from src.planner.phase02_visibility import (
    PHASE_ARTIFACT_STEM as PHASE02_ARTIFACT_STEM,
    generate_visibility_artifacts,
    load_visibility_artifacts,
    save_visibility_artifacts,
    validate_visibility_artifacts,
)
from src.planner.phase03_scoring import (
    PHASE_ARTIFACT_STEM,
    PHASE_NAME,
    decode_configuration_ordinal,
    generate_sparse_score_artifacts,
    get_configuration_dori_scores,
    get_configuration_target_ordinals,
    load_sparse_score_artifacts,
    save_sparse_score_artifacts,
    validate_sparse_score_artifacts,
)
from src.planner.phase03_scoring.core import _build_scoring_constants, _score_distances_to_dori


In [ ]:
def build_floorplan(name: str, rows: list[list[np.int8]], *, grid_cell_size_m: float | None) -> FloorPlanInput:
    grid = np.asarray(rows, dtype=np.int8)
    return FloorPlanInput(
        name=name,
        source_path=Path(f"{name}.png"),
        grid=grid,
        height=int(grid.shape[0]),
        width=int(grid.shape[1]),
        null_cell_count=int(np.count_nonzero(grid == NULL_CELL)),
        open_cell_count=int(np.count_nonzero(grid == OPEN_CELL)),
        solid_cell_count=int(np.count_nonzero(grid == SOLID_CELL)),
        grid_cell_size_m=grid_cell_size_m,
    )


def build_small_workspace_artifacts(grid_cell_size_m: float = 1.0):
    floorplan = build_floorplan(
        "phase03-small",
        [
            [NULL_CELL, NULL_CELL, NULL_CELL, NULL_CELL],
            [NULL_CELL, OPEN_CELL, OPEN_CELL, NULL_CELL],
            [NULL_CELL, OPEN_CELL, OPEN_CELL, NULL_CELL],
            [NULL_CELL, NULL_CELL, NULL_CELL, NULL_CELL],
        ],
        grid_cell_size_m=grid_cell_size_m,
    )
    config = PlannerConfig(
        floorplan_name="phase03-small",
        camera_horizontal_resolution_px=1000,
        camera_horizontal_fov_deg=90.0,
        orientation_step_deg=90,
        k_values=(1,),
    )
    phase01 = generate_candidate_generation_artifacts(floorplan)
    phase02 = generate_visibility_artifacts(floorplan, phase01)
    phase03 = generate_sparse_score_artifacts(floorplan, config, phase01, phase02)
    return floorplan, config, phase01, phase02, phase03


In [ ]:
# Locked angle convention and wrap-around behavior.
small_floorplan, small_config, small_phase01, small_phase02, small_phase03 = build_small_workspace_artifacts()
small_candidate_coords = small_phase01.candidate_cell_coords_rc
small_open_coords = small_phase01.open_cell_coords_rc

np.testing.assert_array_equal(small_phase03.orientation_angles_deg, np.array([0.0, 90.0, 180.0, 270.0], dtype=np.float32))
assert decode_configuration_ordinal(small_phase03, 0) == (0, 0, 0.0)
assert decode_configuration_ordinal(small_phase03, 3) == (0, 3, 270.0)
assert decode_configuration_ordinal(small_phase03, 5) == (1, 1, 90.0)

# Candidate ordinal 0 is the top-left open cell. With a 90-degree FOV, east includes east and southeast,
# while south includes south and southeast because the FOV boundary is inclusive.
np.testing.assert_array_equal(get_configuration_target_ordinals(small_phase03, 0), np.array([1, 3], dtype=np.int32))
np.testing.assert_array_equal(get_configuration_dori_scores(small_phase03, 0), np.array([4, 4], dtype=np.int8))
np.testing.assert_array_equal(get_configuration_target_ordinals(small_phase03, 1), np.empty(0, dtype=np.int32))
np.testing.assert_array_equal(get_configuration_target_ordinals(small_phase03, 3), np.array([2, 3], dtype=np.int32))

# Exact DORI threshold boundaries for a 90-degree camera FOV.
small_scoring_constants = _build_scoring_constants(small_config, small_floorplan.grid_cell_size_m or 1.0)
threshold_distances_m = np.asarray(
    [
        small_config.camera_horizontal_resolution_px / (2.0 * small_config.dori_thresholds.detection),
        small_config.camera_horizontal_resolution_px / (2.0 * small_config.dori_thresholds.observation),
        small_config.camera_horizontal_resolution_px / (2.0 * small_config.dori_thresholds.recognition),
        small_config.camera_horizontal_resolution_px / (2.0 * small_config.dori_thresholds.identification),
    ],
    dtype=np.float64,
)
np.testing.assert_array_equal(_score_distances_to_dori(threshold_distances_m, small_scoring_constants), np.array([1, 2, 3, 4], dtype=np.int8))

"synthetic scoring checks passed"


In [ ]:
workspace = load_workspace()
if workspace.floorplan.grid_cell_size_m is None or workspace.floorplan.grid_cell_size_m <= 0:
    raise ValueError(
        "Phase 03 real-workspace scoring is blocked until the traced floor-plan metadata provides a positive `grid_cell_size_m`."
    )

phase01_artifact_path = workspace.artifact_dir / f"{PHASE01_ARTIFACT_STEM}.npz"
if phase01_artifact_path.exists():
    phase01_artifacts = load_candidate_generation_artifacts(phase01_artifact_path)
else:
    phase01_artifacts = generate_candidate_generation_artifacts(workspace.floorplan)
    save_candidate_generation_artifacts(phase01_artifact_path, phase01_artifacts)
validate_candidate_generation_artifacts(workspace.floorplan, phase01_artifacts)

phase02_artifact_path = workspace.artifact_dir / f"{PHASE02_ARTIFACT_STEM}.npz"
if phase02_artifact_path.exists():
    phase02_artifacts = load_visibility_artifacts(phase02_artifact_path)
else:
    phase02_artifacts = generate_visibility_artifacts(workspace.floorplan, phase01_artifacts)
    save_visibility_artifacts(phase02_artifact_path, phase02_artifacts)
validate_visibility_artifacts(workspace.floorplan, phase01_artifacts, phase02_artifacts)

phase03_artifacts = generate_sparse_score_artifacts(
    workspace.floorplan,
    workspace.config,
    phase01_artifacts,
    phase02_artifacts,
)
phase03_artifact_path = workspace.artifact_dir / f"{PHASE_ARTIFACT_STEM}.npz"
save_sparse_score_artifacts(phase03_artifact_path, phase03_artifacts)
reloaded = load_sparse_score_artifacts(phase03_artifact_path)
validate_sparse_score_artifacts(
    workspace.floorplan,
    workspace.config,
    phase01_artifacts,
    phase02_artifacts,
    reloaded,
)

PHASE_NAME, workspace.floorplan.name, phase03_artifact_path


In [ ]:
score_counts = np.diff(reloaded.score_configuration_offsets)
base_visible_pair_count = int(len(phase02_artifacts.los_target_ordinals))
summary = {
    "floorplan_name": workspace.floorplan.name,
    "grid_shape": workspace.floorplan.shape,
    "grid_cell_size_m": float(workspace.floorplan.grid_cell_size_m),
    "open_target_count": int(reloaded.open_cell_count),
    "candidate_count": int(reloaded.candidate_count),
    "orientation_count": int(len(reloaded.orientation_angles_deg)),
    "configuration_count": int(len(reloaded.configuration_candidate_ordinals)),
    "los_positive_pair_count": base_visible_pair_count,
    "scored_pair_count": int(len(reloaded.score_target_ordinals)),
    "mean_scored_targets_per_configuration": float(score_counts.mean()) if len(score_counts) else 0.0,
    "max_scored_targets_per_configuration": int(score_counts.max()) if len(score_counts) else 0,
}
summary


In [ ]:
candidate_score_counts = score_counts.reshape(reloaded.candidate_count, len(reloaded.orientation_angles_deg))
sample_candidate_ordinals = np.argsort(candidate_score_counts.max(axis=1))[-3:]
open_coords = phase01_artifacts.open_cell_coords_rc
candidate_coords = phase01_artifacts.candidate_cell_coords_rc
dori_palette = {1: "#2563eb", 2: "#16a34a", 3: "#9333ea", 4: "#c2410c"}

for candidate_ordinal in sample_candidate_ordinals.tolist():
    best_angle_ordinal = int(np.argmax(candidate_score_counts[int(candidate_ordinal)]))
    configuration_ordinal = candidate_ordinal * len(reloaded.orientation_angles_deg) + best_angle_ordinal
    _, _, orientation_deg = decode_configuration_ordinal(reloaded, configuration_ordinal)
    target_ordinals = get_configuration_target_ordinals(reloaded, configuration_ordinal)
    score_values = get_configuration_dori_scores(reloaded, configuration_ordinal)
    target_coords = open_coords[target_ordinals]
    candidate_coord = candidate_coords[int(candidate_ordinal)]

    figure, axis = plt.subplots(figsize=(8, 6))
    workspace.floorplan.plot(
        ax=axis,
        title="",
        show_colorbar=False,
    )
    axis.scatter(
        [candidate_coord[1]],
        [candidate_coord[0]],
        s=20,
        c="#111827",
        marker="s",
        linewidths=0,
    )

    for dori_score, color in dori_palette.items():
        score_mask = score_values == dori_score
        if not np.any(score_mask):
            continue
        axis.scatter(
            target_coords[score_mask, 1],
            target_coords[score_mask, 0],
            s=3,
            c=color,
            alpha=0.7,
            linewidths=0,
        )

    wedge_radius = max(10.0, float(np.sqrt(len(target_ordinals) + 1.0)) * 2.0)
    display_orientation_deg = 360.0 - float(orientation_deg)
    display_theta1 = display_orientation_deg - (workspace.config.camera_horizontal_fov_deg / 2.0)
    display_theta2 = display_orientation_deg + (workspace.config.camera_horizontal_fov_deg / 2.0)
    axis.add_patch(
        Wedge(
            (float(candidate_coord[1]), float(candidate_coord[0])),
            wedge_radius,
            display_theta1,
            display_theta2,
            facecolor="#f59e0b",
            alpha=0.18,
            edgecolor="#d97706",
            linewidth=1.0,
        )
    )
    axis.set_title(
        f"{workspace.floorplan.name}: candidate {candidate_ordinal}, orientation {orientation_deg:.0f} deg\n"
        f"scored targets {len(target_ordinals)}"
    )
    figure.tight_layout()
    plt.show()

    score_labels = {1: "Detection", 2: "Observation", 3: "Recognition", 4: "Identification"}
    histogram_scores = np.array(sorted(score_labels), dtype=np.int32)
    histogram_counts = np.array([(score_values == score).sum() for score in histogram_scores], dtype=np.int64)

    figure, axis = plt.subplots(figsize=(8, 4))
    axis.bar(
        [score_labels[int(score)] for score in histogram_scores],
        histogram_counts,
        color=[dori_palette[int(score)] for score in histogram_scores],
    )
    axis.set_title(
        f"{workspace.floorplan.name}: candidate {candidate_ordinal}, orientation {orientation_deg:.0f} deg score histogram"
    )
    axis.set_xlabel("DORI score")
    axis.set_ylabel("Target count")
    figure.tight_layout()
    plt.show()
